# Test Suite Functionality

This notebook helps you learn and run the project test suite.

It covers:

- test file inventory
- mapping tests to functionality areas
- focused pytest commands
- optional full suite run
- reading pass/fail/skip output

In [ ]:
from pathlib import Path
import ast
import subprocess

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / 'tests').exists()), cwd)
tests_dir = project_root / 'tests'
print('project_root:', project_root)
print('tests_dir:', tests_dir)

## 1. Test Inventory

In [ ]:
test_files = sorted(tests_dir.glob('test*.py'))
print('test files:', len(test_files))
for p in test_files:
    print('-', p.name)

## 2. Test Classes and Functions

This parses test files without importing them, so it is safe even if dependencies are missing.

In [ ]:
inventory = []
for p in test_files:
    tree = ast.parse(p.read_text(encoding='utf-8', errors='replace'))
    classes = [n.name for n in tree.body if isinstance(n, ast.ClassDef)]
    funcs = [n.name for n in tree.body if isinstance(n, ast.FunctionDef) and n.name.startswith('test_')]
    methods = []
    for cls in [n for n in tree.body if isinstance(n, ast.ClassDef)]:
        for item in cls.body:
            if isinstance(item, ast.FunctionDef) and item.name.startswith('test_'):
                methods.append(f'{cls.name}.{item.name}')
    inventory.append((p.name, classes, funcs, methods))

for filename, classes, funcs, methods in inventory:
    print('\n' + filename)
    print(' classes:', classes)
    for name in funcs + methods:
        print('  -', name)

## 3. Functionality to Test File Map

In [ ]:
mapping = {
    'InformationAgent': ['tests/test_information_agent.py'],
    'KnowledgeAgent': ['tests/test_knowledge_agent.py'],
    'MetadataAgent': ['tests/test_metadata_agent.py'],
    'CapacityAgent': ['tests/test_capacity_agent.py'],
    'RuleAgent': ['tests/test_rule_agent.py'],
    'Redis/cache/LLM factory/graph cache': ['tests/test_day14.py'],
    'Retry + HITL + graph resilience': ['tests/test_day15.py'],
    'FastAPI + SSE + token guard': ['tests/test_day16.py'],
    'Teams bot + Adaptive Cards': ['tests/test17_day17.py'],
}

for area, files in mapping.items():
    print(f'{area}:')
    for f in files:
        print('  ', f, 'exists=', (project_root / f).exists())

## 4. Focused Test Commands

Run these from project root when debugging a specific area.

In [ ]:
commands = [
    'uv run pytest tests/test_information_agent.py -q',
    'uv run pytest tests/test_knowledge_agent.py -q',
    'uv run pytest tests/test_metadata_agent.py -q',
    'uv run pytest tests/test_capacity_agent.py -q',
    'uv run pytest tests/test_rule_agent.py -q',
    'uv run pytest tests/test_day14.py -q',
    'uv run pytest tests/test_day15.py -q',
    'uv run pytest tests/test_day16.py -q',
    'uv run pytest tests/test17_day17.py -q',
    'uv run pytest tests -q',
]
for cmd in commands:
    print(cmd)

## 5. Optional Focused Run

Set `RUN_TESTS = True` to run a focused test command from inside this notebook.

In [ ]:
RUN_TESTS = False
TEST_COMMAND = ['uv', 'run', 'pytest', 'tests/test_day16.py', '-q']

if RUN_TESTS:
    result = subprocess.run(TEST_COMMAND, cwd=project_root, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)
    print('returncode:', result.returncode)
else:
    print('Skipped. Set RUN_TESTS = True to execute:', ' '.join(TEST_COMMAND))